# defence

> Phase 9 defences: private trust, shared reputation and graduated sanctions (Ostrom) with spot-check monitoring, for agents on the blackboard and message channels.

Spec: `specs/multiagent/phase9/` (design-low-level section 3).

In [ ]:
#| default_exp defence

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from collections import deque

import numpy as np

from explRL.adversary import ReportLog
from explRL.harness import Blackboard, MessageBus
from explRL.harness_agents import CommBandit


class Monitor:
    """Shared reputation `rho` per sender, updated by every agent's checks; with `sanctions`, strikes (checks with
    error above `theta`) count for `window` seconds and set the sanction level: 1 warning, 2 half weight, 3 revoked."""
    def __init__(self, n, sanctions=False, theta=1.0, window=120.0, alpha=0.1, sigma=0.5, channel=None):
        self.n, self.sanctions, self.theta, self.window = n, bool(sanctions), float(theta), float(window)
        self.alpha, self.sigma, self.channel = float(alpha), float(sigma), channel
        self.rho, self.strikes = np.ones(n), [deque() for _ in range(n)]
        self.maxLevel, self.firstRevoke = np.zeros(n, int), np.full(n, np.nan)

    def record(self, s, error, t):
        self.rho[s] = (1 - self.alpha) * self.rho[s] + self.alpha * np.exp(-error / self.sigma)
        if self.sanctions:
            if error > self.theta:
                self.strikes[s].append(t)
            self._sync(t)

    def level(self, s, t):
        q = self.strikes[s]
        while q and q[0] <= t - self.window:
            q.popleft()
        return min(len(q), 3)

    def _sync(self, t):
        lv = np.array([self.level(s, t) for s in range(self.n)])
        self.maxLevel = np.maximum(self.maxLevel, lv)
        self.firstRevoke[(lv == 3) & np.isnan(self.firstRevoke)] = t
        if self.channel is not None:
            self.channel.blocked = {int(s) for s in np.flatnonzero(lv == 3)}
        return lv

    def weights(self, t):
        if not self.sanctions:
            return self.rho
        return self.rho * np.array([1.0, 1.0, 0.5, 0.0])[self._sync(t)]


class DefendedBandit(ReportLog, CommBandit):
    """`CommBandit` with a defence: reads reports weighted per sender ('trust': its own `tau`; 'reputation',
    'sanctions': the `Monitor`'s weights), checks senders' reports after each own stay, and with probability `q`
    per event makes a spot-check trip. With 'none' and q = 0 it chooses exactly as `CommBandit`; it always logs reports."""
    def __init__(self, channel=None, defence='none', monitor=None, q=0.0, rng=None, alpha=0.1, sigma=0.5, **kw):
        super().__init__(channel, **kw)
        self.defence, self.monitor, self.q, self.rng = defence, monitor, float(q), rng
        self.alpha, self.sigma = float(alpha), float(sigma)
        self.tau = self.msgs = self.lastStay = None
        self.checks = self.deliberate = 0
        self.checkSeconds, self._trip, self._onTrip = 0.0, None, False

    def _setup(self, world):
        if self.tau is None:
            self.tau, self.lastStay = np.ones(world.nAgents), np.full(world.L, -np.inf)
            self.msgs = [deque() for _ in range(world.L)]
        if self.rates is None:                                          # a first-event spot check skips banditAgent.act
            self.rates = [deque(maxlen=self.K) for _ in range(world.L)]

    def event(self, world, t):
        self._world, self._t = world, t
        self._setup(world)
        if isinstance(self.channel, MessageBus):                       # store for CommBandit and with senders
            for s, j, v, tm in self.channel.take(self.id):
                if self.w != 0:
                    self._store()[j].append(v)
                self.msgs[j].append((s, v, tm))
        return super().event(world, t)

    def _weights(self):
        if self.defence == 'none':
            return None
        return self.tau if self.defence == 'trust' else self.monitor.weights(self._t)

    def _msgSums(self, wts=None):
        S, C = np.zeros(len(self.msgs)), np.zeros(len(self.msgs))
        for j, dq in enumerate(self.msgs):
            for s, v, _ in dq:
                f = 1.0 if wts is None else wts[s]
                S[j] += f * v
                C[j] += f
        return S, C

    def _stats(self, obs):
        wts = self._weights()
        if wts is None or self.channel is None:
            return super()._stats(obs)
        if self.rates is None:
            self.rates = [deque(maxlen=self.K) for _ in range(self._world.L)]
        S = np.array([sum(r) for r in self.rates], float)
        n = np.array([len(r) for r in self.rates], float)
        Sv, Cv = self.channel.viewBy(self.id, self._t, wts) if isinstance(self.channel, Blackboard) else self._msgSums(wts)
        S, n = S + self.w * Sv, n + self.w * Cv
        return np.where(n > 0, S / np.where(n > 0, n, 1), 0.0), n

    def _checkTarget(self):
        "Highest reported location (others' reports, unweighted) not stayed at in the last Tmove; None if none."
        S, C = self.channel.view(self.id, self._t) if isinstance(self.channel, Blackboard) else self._msgSums()
        rep = np.where(C > 0, S / np.where(C > 0, C, 1), -np.inf)
        ok = (C > 0) & (self.lastStay < self._t - (self._world.Tmove or np.inf))
        return int(np.flatnonzero(ok)[np.argmax(rep[ok])]) if ok.any() else None

    def act(self, obs):
        loc = obs['location']
        if self._trip is not None and self._trip == loc:                # the checking stay at the end of a trip
            self._trip, self._onTrip = None, True
            return loc
        self._trip, self._onTrip = None, False
        if self.q > 0 and self.channel is not None and self.rng.random() < self.q:
            j = self._checkTarget()
            if j is not None and j != loc:
                self._trip, self._onTrip = j, True
                self.deliberate += 1
                return j
        return super().act(obs)

    def learn(self, obs, target, reward, info):
        bus = self.channel if isinstance(self.channel, MessageBus) else None
        before = (bus.stats[self.id]['sent'], bus.stats[self.id].get('rejected', 0)) if bus else None
        super().learn(obs, target, reward, info)                        # own stays, honest write or send
        if self._onTrip:
            self.checkSeconds += info['dur']
        if info['kind'] != 'stay':
            return
        loc, r = obs['location'], reward / info['dur']
        self.lastStay[loc] = self._t
        if isinstance(self.channel, Blackboard):
            self._log(loc, r)
        elif bus and (bus.stats[self.id]['sent'], bus.stats[self.id].get('rejected', 0)) != before:
            v, j = max(((np.mean(q), j) for j, q in enumerate(self.rates) if len(q)), key=lambda x: (x[0], -x[1]))
            self._log(j, v)
        if self.defence != 'none':
            self._check(loc, r)

    def _check(self, j, r):
        "Compare every other sender's current report at j with the observed rate r; update trust or the monitor."
        self.checks += 1
        if isinstance(self.channel, Blackboard):
            b = self.channel
            b.view(self.id, self._t)                                     # fade the sums to now
            has = b.Ci[:, j] > 0.05
            has[self.id] = False
            reps = dict(zip(np.flatnonzero(has), b.Si[has, j] / b.Ci[has, j]))
        else:
            horizon = self._t - (self._world.Tmove or np.inf)
            reps = {s: v for s, v, tm in self.msgs[j] if tm >= horizon and s != self.id}
        for s, m in reps.items():
            e = abs(m - r)
            if self.defence == 'trust':
                self.tau[s] = (1 - self.alpha) * self.tau[s] + self.alpha * np.exp(-e / self.sigma)
            else:
                self.monitor.record(int(s), e, self._t)

## Tests

In [ ]:
import json
import numpy as np
from fastcore.test import test_eq, test_close
from explRL.defence import Monitor, DefendedBandit
from explRL.harness import Blackboard, MessageBus
from explRL.harness_agents import CommBandit
from explRL.harness_runs import benchWorld
from explRL.multi import runTeam, spawnRngs

# one trust update matches the formula; a sender whose reports are always wrong loses weight
w = benchWorld(); w.reset(0, 3)
bb = Blackboard(w.L, n=3)
bb.write(1, 0, 0.5, 0.0); bb.write(2, 0, -2.6, 0.0)
db = DefendedBandit(bb, 'trust'); db.id, db._world, db._t = 0, w, 1.0; db._setup(w)
db._check(0, 0.85)
test_close(db.tau[1], 0.9 + 0.1 * np.exp(-0.35 / 0.5)); test_close(db.tau[2], 0.9 + 0.1 * np.exp(-3.45 / 0.5))
for k in range(30): db._t = 2.0 + k; db._check(0, 0.85)
assert db.tau[2] < 0.1 < 0.5 < db.tau[1]

# sanctions: strikes within 120 s raise the level (warning, half weight, revoked); the channel then drops the
# sender's writes; once the strikes leave the window the sender is reinstated
mon = Monitor(3, sanctions=True, theta=1.0, channel=bb)
mon.record(1, 2.0, 0.0); test_eq(mon.level(1, 0.0), 1); test_close(mon.weights(0.0)[1], mon.rho[1])
mon.record(1, 2.0, 10.0); test_close(mon.weights(10.0)[1], 0.5 * mon.rho[1])
mon.record(1, 0.2, 15.0); test_eq(mon.level(1, 15.0), 2)                 # a good check is not a strike
mon.record(1, 2.0, 20.0); test_eq(mon.level(1, 20.0), 3); test_eq(bb.blocked, {1}); test_eq(mon.weights(20.0)[1], 0.0)
bb.write(1, 3, 1.0, 21.0); test_eq(bb.stats[1]['rejected'], 1)
test_eq(mon.firstRevoke[1], 20.0); test_eq(int(mon.maxLevel[1]), 3)
mon.weights(141.0); test_eq(bb.blocked, set()); test_eq(mon.level(1, 141.0), 0)  # all strikes older than 120 s
assert np.isnan(mon.firstRevoke[0])
rep = Monitor(3); rep.record(2, 3.0, 0.0); test_close(rep.weights(0.0)[2], 0.9 + 0.1 * np.exp(-6.0))

# with 'none' and q = 0 a DefendedBandit chooses exactly as CommBandit (blackboard and messages), and still logs
for kind in ('blackboard', 'messages'):
    hs, logs = [], []
    for cls in (CommBandit, DefendedBandit):
        wr = benchWorld(); wr.reset(4, 8)
        ch = Blackboard(wr.L, n=8) if kind == 'blackboard' else MessageBus(8, rngs=spawnRngs(2004, 8))
        team = [cls(ch) for _ in range(8)]
        hs.append(runTeam(team, wr, 200.0).records)
        logs.append(sum(len(getattr(m, 'reportLog', [])) for m in team))
    test_eq(hs[0], hs[1]); assert logs[1] > 0
    assert DefendedBandit(None).whiteScore() != DefendedBandit(None).whiteScore()   # nan: never reported

# honest reports sit close to own beliefs (white-box score), deliberate checks happen at most at rate q and cost time
wr = benchWorld(); wr.reset(5, 8)
ch = Blackboard(wr.L, n=8); mon = Monitor(8, sanctions=True, channel=ch)
team = [DefendedBandit(ch, 'sanctions', mon, q=0.2, rng=r) for r in spawnRngs(3005, 8)]
h = runTeam(team, wr, 300.0)
draws = sum(1 for r in h.records)                                          # upper bound on decisions
d = sum(m.deliberate for m in team); assert 0 < d < 0.2 * draws
assert sum(m.checkSeconds for m in team) > 0 and sum(m.checks for m in team) > 0
assert max(m.whiteScore() for m in team) < 1.5

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()